In [1]:
# ============================================================
# MODEL: HISTOGRAM GRADIENT BOOSTING
# 5-FOLD CROSS VALIDATION
# PHISHING URL DETECTION
# ============================================================

import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate


# ============================================================
# STEP 1: LOAD DATASET
# ============================================================

df = pd.read_csv("phishing_website_cleaned.csv")

print("=" * 70)
print("HISTOGRAM GRADIENT BOOSTING - 5-FOLD CROSS VALIDATION")
print("=" * 70)

print("\nDataset shape:", df.shape)


# ============================================================
# STEP 2: SELECT FEATURES
# ============================================================

URL_FEATURES = [
    'qty_dot_url',
    'qty_hyphen_url',
    'qty_underline_url',
    'qty_slash_url',
    'qty_questionmark_url',
    'qty_equal_url',
    'qty_at_url',
    'qty_and_url',
    'qty_exclamation_url',
    'qty_space_url',
    'qty_tilde_url',
    'qty_comma_url',
    'qty_plus_url',
    'qty_asterisk_url',
    'qty_hashtag_url',
    'qty_dollar_url',
    'qty_percent_url',
    'length_url',

    'qty_dot_domain',
    'qty_hyphen_domain',
    'qty_underline_domain',
    'qty_slash_domain',
    'qty_questionmark_domain',
    'qty_equal_domain',
    'qty_at_domain',
    'qty_and_domain',
    'qty_exclamation_domain',
    'qty_space_domain',
    'qty_tilde_domain',
    'qty_comma_domain',
    'qty_plus_domain',
    'qty_asterisk_domain',
    'qty_hashtag_domain',
    'qty_dollar_domain',
    'qty_percent_domain',

    'qty_vowels_domain',
    'domain_length',
    'domain_in_ip',
    'server_client_domain',
    'email_in_url',
    'tls_ssl_certificate',
    'url_shortened'
]


# ============================================================
# STEP 3: CHECK AVAILABLE FEATURES
# ============================================================

available_features = [
    col for col in URL_FEATURES
    if col in df.columns
]

missing_features = [
    col for col in URL_FEATURES
    if col not in df.columns
]

print("\n" + "=" * 70)
print("FEATURE INFORMATION")
print("=" * 70)

print("\nAvailable features:", len(available_features))

if missing_features:

    print("\nMissing features:")

    for feature in missing_features:
        print("-", feature)


# ============================================================
# STEP 4: CREATE X AND y
# ============================================================

X = df[available_features].copy()
y = df["phishing"].copy()

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)


# ============================================================
# STEP 5: REMOVE CONSTANT FEATURES
# ============================================================

constant_features = [
    col
    for col in X.columns
    if X[col].nunique() <= 1
]

if constant_features:

    print("\n" + "=" * 70)
    print("CONSTANT FEATURES REMOVED")
    print("=" * 70)

    for feature in constant_features:
        print("-", feature)

    X = X.drop(columns=constant_features)

else:

    print("\nNo constant features found.")


print("\nFinal features shape:", X.shape)


# ============================================================
# STEP 6: CHECK TARGET
# ============================================================

print("\n" + "=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)

print("\nTarget values:")

print(y.value_counts())

print("\nTarget missing values:", y.isna().sum())


# ============================================================
# STEP 7: REMOVE ROWS WITH MISSING TARGET
# ============================================================

if y.isna().sum() > 0:

    valid_rows = y.notna()

    X = X.loc[valid_rows].copy()
    y = y.loc[valid_rows].copy()

    print("\nRows with missing target removed.")

print("\nFinal X shape:", X.shape)
print("Final y shape:", y.shape)


# ============================================================
# STEP 8: CREATE 5-FOLD STRATIFIED CROSS VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("\n" + "=" * 70)
print("5-FOLD CROSS VALIDATION")
print("=" * 70)

print("\nNumber of folds:", cv.n_splits)
print("Sampling method: None")
print("Shuffle: True")
print("Random state: 42")


# ============================================================
# STEP 9: CREATE HISTOGRAM GRADIENT BOOSTING MODEL
# ============================================================

model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)


# ============================================================
# STEP 10: CROSS VALIDATION
# ============================================================

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

cv_results = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)


# ============================================================
# STEP 11: DISPLAY EACH FOLD RESULT
# ============================================================

print("\n" + "=" * 70)
print("INDIVIDUAL FOLD RESULTS")
print("=" * 70)

for i in range(5):

    print(f"\nFold {i + 1}")

    print(
        "Accuracy :",
        f"{cv_results['test_accuracy'][i] * 100:.2f}%"
    )

    print(
        "Precision:",
        f"{cv_results['test_precision'][i] * 100:.2f}%"
    )

    print(
        "Recall   :",
        f"{cv_results['test_recall'][i] * 100:.2f}%"
    )

    print(
        "F1-Score :",
        f"{cv_results['test_f1'][i] * 100:.2f}%"
    )


# ============================================================
# STEP 12: CALCULATE MEAN AND STANDARD DEVIATION
# ============================================================

accuracy_mean = cv_results["test_accuracy"].mean()
accuracy_std = cv_results["test_accuracy"].std()

precision_mean = cv_results["test_precision"].mean()
precision_std = cv_results["test_precision"].std()

recall_mean = cv_results["test_recall"].mean()
recall_std = cv_results["test_recall"].std()

f1_mean = cv_results["test_f1"].mean()
f1_std = cv_results["test_f1"].std()


# ============================================================
# STEP 13: DISPLAY FINAL CROSS VALIDATION RESULTS
# ============================================================

print("\n" + "=" * 70)
print("5-FOLD CROSS VALIDATION SUMMARY")
print("=" * 70)

print(
    "\nAccuracy :",
    f"{accuracy_mean * 100:.2f}% ± {accuracy_std * 100:.2f}%"
)

print(
    "Precision:",
    f"{precision_mean * 100:.2f}% ± {precision_std * 100:.2f}%"
)

print(
    "Recall   :",
    f"{recall_mean * 100:.2f}% ± {recall_std * 100:.2f}%"
)

print(
    "F1-Score :",
    f"{f1_mean * 100:.2f}% ± {f1_std * 100:.2f}%"
)


# ============================================================
# STEP 14: TRAIN FINAL MODEL ON FULL DATASET
# ============================================================

final_model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

final_model.fit(
    X,
    y
)


# ============================================================
# STEP 15: FINAL MODEL INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL MODEL")
print("=" * 70)

print("\n✓ Final Histogram Gradient Boosting model trained")
print("✓ Training data:", X.shape)
print("✓ Sampling method: None")
print("✓ Cross-validation: 5-Fold Stratified")
print("\nModel is ready for further prediction.")
print("=" * 70)

HISTOGRAM GRADIENT BOOSTING - 5-FOLD CROSS VALIDATION

Dataset shape: (9944, 56)

FEATURE INFORMATION

Available features: 42

Features shape: (9944, 42)
Target shape: (9944,)

CONSTANT FEATURES REMOVED
- qty_hashtag_url
- qty_slash_domain
- qty_questionmark_domain
- qty_equal_domain
- qty_at_domain
- qty_and_domain
- qty_exclamation_domain
- qty_space_domain
- qty_tilde_domain
- qty_comma_domain
- qty_plus_domain
- qty_asterisk_domain
- qty_hashtag_domain
- qty_dollar_domain
- qty_percent_domain

Final features shape: (9944, 27)

TARGET DISTRIBUTION

Target values:
phishing
0    6491
1    3453
Name: count, dtype: int64

Target missing values: 0

Final X shape: (9944, 27)
Final y shape: (9944,)

5-FOLD CROSS VALIDATION

Number of folds: 5
Sampling method: None
Shuffle: True
Random state: 42

INDIVIDUAL FOLD RESULTS

Fold 1
Accuracy : 92.61%
Precision: 89.08%
Recall   : 89.73%
F1-Score : 89.40%

Fold 2
Accuracy : 93.06%
Precision: 89.56%
Recall   : 90.59%
F1-Score : 90.07%

Fold 3
Accur

In [2]:
# ============================================================
# MODEL: HISTOGRAM GRADIENT BOOSTING
# PHISHING URL DETECTION
# ============================================================

import pandas as pd
import re
from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# STEP 1: LOAD DATASET
# ============================================================

df = pd.read_csv("phishing_website_cleaned.csv")

print("=" * 70)
print("HISTOGRAM GRADIENT BOOSTING PHISHING URL DETECTION")
print("=" * 70)

print("\nDataset shape:", df.shape)


# ============================================================
# STEP 2: SELECT FEATURES
# ============================================================

URL_FEATURES = [
    'qty_dot_url',
    'qty_hyphen_url',
    'qty_underline_url',
    'qty_slash_url',
    'qty_questionmark_url',
    'qty_equal_url',
    'qty_at_url',
    'qty_and_url',
    'qty_exclamation_url',
    'qty_space_url',
    'qty_tilde_url',
    'qty_comma_url',
    'qty_plus_url',
    'qty_asterisk_url',
    'qty_hashtag_url',
    'qty_dollar_url',
    'qty_percent_url',
    'length_url',

    'qty_dot_domain',
    'qty_hyphen_domain',
    'qty_underline_domain',
    'qty_slash_domain',
    'qty_questionmark_domain',
    'qty_equal_domain',
    'qty_at_domain',
    'qty_and_domain',
    'qty_exclamation_domain',
    'qty_space_domain',
    'qty_tilde_domain',
    'qty_comma_domain',
    'qty_plus_domain',
    'qty_asterisk_domain',
    'qty_hashtag_domain',
    'qty_dollar_domain',
    'qty_percent_domain',

    'qty_vowels_domain',
    'domain_length',
    'domain_in_ip',
    'server_client_domain',
    'email_in_url',
    'tls_ssl_certificate',
    'url_shortened'
]


# ============================================================
# STEP 3: CHECK FEATURES
# ============================================================

available_features = [
    col for col in URL_FEATURES
    if col in df.columns
]

missing_features = [
    col for col in URL_FEATURES
    if col not in df.columns
]

print("\n" + "=" * 70)
print("FEATURE INFORMATION")
print("=" * 70)

print("\nAvailable features:", len(available_features))

if missing_features:
    print("\nMissing features:")
    for feature in missing_features:
        print("-", feature)


# ============================================================
# STEP 4: CREATE X AND y
# ============================================================

X = df[available_features].copy()
y = df["phishing"].copy()

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)


# ============================================================
# STEP 5: REMOVE CONSTANT FEATURES
# ============================================================

constant_features = [
    col for col in X.columns
    if X[col].nunique() <= 1
]

if constant_features:

    print("\n" + "=" * 70)
    print("CONSTANT FEATURES REMOVED")
    print("=" * 70)

    for feature in constant_features:
        print("-", feature)

    X = X.drop(columns=constant_features)

else:

    print("\nNo constant features found.")


print("\nFinal features shape:", X.shape)


# ============================================================
# STEP 6: TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n" + "=" * 70)
print("STEP 1: TRAIN / TEST SPLIT")
print("=" * 70)

print("\nTraining data:", X_train.shape)
print("Testing data :", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())


# ============================================================
# STEP 7: TRAIN HISTOGRAM GRADIENT BOOSTING
# ============================================================

hist_model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

hist_model.fit(
    X_train,
    y_train
)

print("\n" + "=" * 70)
print("STEP 2: MODEL TRAINING")
print("=" * 70)

print("\n✓ Histogram Gradient Boosting model trained successfully")


# ============================================================
# STEP 8: PREDICTION
# ============================================================

y_pred = hist_model.predict(
    X_test
)

print("\n" + "=" * 70)
print("STEP 3: PREDICTION")
print("=" * 70)

print("\ny_test shape:", y_test.shape)
print("y_pred shape:", y_pred.shape)

print("\n✓ Prediction completed successfully")


# ============================================================
# STEP 9: MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\n" + "=" * 70)
print("STEP 4: MODEL EVALUATION")
print("=" * 70)

print(
    "\nAccuracy:",
    f"{accuracy * 100:.2f}%"
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Legitimate",
            "Phishing"
        ]
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


# ============================================================
# STEP 10: URL FEATURE EXTRACTION
# ============================================================

def extract_url_features(url):

    url = url.strip()

    # Handle markdown-style URL
    if url.startswith("[") and "](" in url:

        url = url.split("](", 1)[1]

        if url.endswith(")"):
            url = url[:-1]

    # Add protocol if missing
    if not url.startswith(("http://", "https://")):
        url = "http://" + url

    parsed = urlparse(url)

    full_url = url

    domain = parsed.netloc.split(":")[0]

    features = {}


    # --------------------------------------------------------
    # URL CHARACTER FEATURES
    # --------------------------------------------------------

    features["qty_dot_url"] = full_url.count(".")
    features["qty_hyphen_url"] = full_url.count("-")
    features["qty_underline_url"] = full_url.count("_")
    features["qty_slash_url"] = full_url.count("/")
    features["qty_questionmark_url"] = full_url.count("?")
    features["qty_equal_url"] = full_url.count("=")
    features["qty_at_url"] = full_url.count("@")
    features["qty_and_url"] = full_url.count("&")
    features["qty_exclamation_url"] = full_url.count("!")
    features["qty_space_url"] = full_url.count(" ")
    features["qty_tilde_url"] = full_url.count("~")
    features["qty_comma_url"] = full_url.count(",")
    features["qty_plus_url"] = full_url.count("+")
    features["qty_asterisk_url"] = full_url.count("*")
    features["qty_hashtag_url"] = full_url.count("#")
    features["qty_dollar_url"] = full_url.count("$")
    features["qty_percent_url"] = full_url.count("%")

    features["length_url"] = len(full_url)


    # --------------------------------------------------------
    # DOMAIN CHARACTER FEATURES
    # --------------------------------------------------------

    features["qty_dot_domain"] = domain.count(".")
    features["qty_hyphen_domain"] = domain.count("-")
    features["qty_underline_domain"] = domain.count("_")
    features["qty_slash_domain"] = domain.count("/")
    features["qty_questionmark_domain"] = domain.count("?")
    features["qty_equal_domain"] = domain.count("=")
    features["qty_at_domain"] = domain.count("@")
    features["qty_and_domain"] = domain.count("&")
    features["qty_exclamation_domain"] = domain.count("!")
    features["qty_space_domain"] = domain.count(" ")
    features["qty_tilde_domain"] = domain.count("~")
    features["qty_comma_domain"] = domain.count(",")
    features["qty_plus_domain"] = domain.count("+")
    features["qty_asterisk_domain"] = domain.count("*")
    features["qty_hashtag_domain"] = domain.count("#")
    features["qty_dollar_domain"] = domain.count("$")
    features["qty_percent_domain"] = domain.count("%")


    # --------------------------------------------------------
    # DOMAIN FEATURES
    # --------------------------------------------------------

    features["qty_vowels_domain"] = sum(
        c.lower() in "aeiou"
        for c in domain
    )

    features["domain_length"] = len(domain)


    # --------------------------------------------------------
    # IP ADDRESS
    # --------------------------------------------------------

    ip_pattern = r"^(\d{1,3}\.){3}\d{1,3}$"

    features["domain_in_ip"] = int(
        bool(re.match(ip_pattern, domain))
    )


    # --------------------------------------------------------
    # SERVER / CLIENT DOMAIN
    # --------------------------------------------------------

    features["server_client_domain"] = int(
        "server" in domain.lower()
        or
        "client" in domain.lower()
    )


    # --------------------------------------------------------
    # EMAIL IN URL
    # --------------------------------------------------------

    features["email_in_url"] = int(
        "@" in full_url
    )


    # --------------------------------------------------------
    # HTTPS
    # --------------------------------------------------------

    features["tls_ssl_certificate"] = int(
        parsed.scheme == "https"
    )


    # --------------------------------------------------------
    # URL SHORTENER
    # --------------------------------------------------------

    shorteners = [
        "bit.ly",
        "tinyurl.com",
        "t.co",
        "goo.gl",
        "is.gd",
        "ow.ly"
    ]

    features["url_shortened"] = int(
        any(
            shortener in domain.lower()
            for shortener in shorteners
        )
    )


    return features


# ============================================================
# STEP 11: PREPARE URL FOR MODEL
# ============================================================

def prepare_url_for_model(url):

    features = extract_url_features(url)

    url_df = pd.DataFrame([features])


    missing_features = [
        col
        for col in X.columns
        if col not in url_df.columns
    ]

    if missing_features:

        raise ValueError(
            "Missing features for prediction: "
            + str(missing_features)
        )


    # Same feature order as training data
    url_df = url_df[X.columns]

    return url_df

HISTOGRAM GRADIENT BOOSTING PHISHING URL DETECTION

Dataset shape: (9944, 56)

FEATURE INFORMATION

Available features: 42

Features shape: (9944, 42)
Target shape: (9944,)

CONSTANT FEATURES REMOVED
- qty_hashtag_url
- qty_slash_domain
- qty_questionmark_domain
- qty_equal_domain
- qty_at_domain
- qty_and_domain
- qty_exclamation_domain
- qty_space_domain
- qty_tilde_domain
- qty_comma_domain
- qty_plus_domain
- qty_asterisk_domain
- qty_hashtag_domain
- qty_dollar_domain
- qty_percent_domain

Final features shape: (9944, 27)

STEP 1: TRAIN / TEST SPLIT

Training data: (7955, 27)
Testing data : (1989, 27)

Training target distribution:
phishing
0    5193
1    2762
Name: count, dtype: int64

Testing target distribution:
phishing
0    1298
1     691
Name: count, dtype: int64

STEP 2: MODEL TRAINING

✓ Histogram Gradient Boosting model trained successfully

STEP 3: PREDICTION

y_test shape: (1989,)
y_pred shape: (1989,)

✓ Prediction completed successfully

STEP 4: MODEL EVALUATION

Accu